# Face variation prompt generation

Generate image-editing prompts for existing face portraits using attribute distributions and **Qwen/Qwen3-14B**. Set the paths and generation options below, provide the required image folders and variation files, and run the notebook from top to bottom. No generated images are created by this notebook.

In [ ]:
from pathlib import Path
import csv
import math
import random
import re

import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Paths are relative to the notebook's working directory; absolute paths also work.
INPUT_FOLDER = Path("generated_images")
VARIATION_DIR = Path("variations")
OUTPUT_DIR = Path("prompts_files")

# Generation settings.
NUM_IMAGES_PER_ID = 16
SEED = 42
MODEL_NAME = "Qwen/Qwen3-14B"
TEMPERATURE = 0.2
MAX_NEW_TOKENS = 180
TOP_P = 0.8
TOP_K = 30
REPETITION_PENALTY = 1.05

# Variation distribution files (CSV-formatted text: label, probability[, prompt text]).
COMMON_VARIATION_FILES = {
    "gender": "genders.txt",
    "pose": "pose.txt",
    "lighting": "lighting.txt",
    "background": "background.txt",
    "clothing": "clothing.txt",
}
GENDER_SPECIFIC_VARIATION_FILES = {
    "male": {
        "hairstyle": "hairstyle_male.txt",
        "headwear": "headwear_male.txt",
        "facial_hair": "facialhair_male.txt",
    },
    "female": {
        "hairstyle": "hairstyle_female.txt",
        "headwear": "headwear_female.txt",
        "makeup": "makeup_female.txt",
        "jewelry": "jewelry_female.txt",
    },
}

# Optional: sample an additional jewelry item with probability 0.25.
MULTI_FEATURE_RULES = {
    "jewelry": {
        "enabled": True,
        "max_items": 2,
        "chance_extra_item": 0.25,
        "avoid_labels": {"No-Jewelry"},
    }
}

random.seed(SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def get_images_for_gender(gender):
    gender_dir = INPUT_FOLDER / f"face_generation_{gender}"
    if not gender_dir.is_dir():
        raise FileNotFoundError(f"Image folder not found: {gender_dir}")
    return sorted(p for p in gender_dir.iterdir()
                  if p.is_file() and p.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"})

identity_counts = {gender: len(get_images_for_gender(gender))
                   for gender in ("male", "female")}
if not any(identity_counts.values()):
    raise ValueError(f"No images found under {INPUT_FOLDER}")

print("Portraits per gender:", identity_counts)

In [ ]:
def slug_to_text(slug):
    return slug.replace("-", " ").replace("_", " ").strip()


def read_probability_file(path):
    items = []
    with path.open("r", encoding="utf-8") as f:
        for row in csv.reader(f):
            if not row or not any(cell.strip() for cell in row):
                continue
            if len(row) < 2:
                raise ValueError(f"Expected label,probability[,text] in {path}: {row}")
            label = row[0].strip()
            try:
                probability = float(row[1].strip())
            except ValueError as exc:
                raise ValueError(f"Invalid probability in {path}: {row}") from exc
            if probability < 0 or not math.isfinite(probability):
                raise ValueError(f"Invalid probability in {path}: {row}")
            text = row[2].strip() if len(row) >= 3 and row[2].strip() else slug_to_text(label)
            items.append({"label": label, "probability": probability, "text": text})
    total = sum(item["probability"] for item in items)
    if total <= 0:
        raise ValueError(f"No positive probabilities in {path}")
    for item in items:
        item["probability"] /= total
    return items

In [ ]:
def load_variation_data(file_map):
    data = {}
    for feature_name, filename in file_map.items():
        path = VARIATION_DIR / filename
        if not path.is_file():
            raise FileNotFoundError(f"Required variation file not found: {path}")
        data[feature_name] = read_probability_file(path)
    return data

common_variation_data = load_variation_data(COMMON_VARIATION_FILES)
gender_specific_variation_data = {
    gender: load_variation_data(file_map)
    for gender, file_map in GENDER_SPECIFIC_VARIATION_FILES.items()
}

In [ ]:
def allocate_by_probability(items, n, seed=None):
    """
    Creates exactly n sampled labels according to probability.
    Uses largest remainder method, then shuffles.

    Example:
    probability 0.4 with n=10 -> about 4 samples.
    """
    rng = random.Random(seed)

    raw_counts = []
    total_floor = 0

    for item in items:
        exact = item["probability"] * n
        floor_count = math.floor(exact)
        remainder = exact - floor_count

        raw_counts.append({
            "item": item,
            "floor_count": floor_count,
            "remainder": remainder
        })

        total_floor += floor_count

    remaining = n - total_floor

    # Give remaining slots to largest remainders
    raw_counts.sort(key=lambda x: x["remainder"], reverse=True)

    for i in range(remaining):
        raw_counts[i]["floor_count"] += 1

    allocated = []

    for entry in raw_counts:
        allocated.extend([entry["item"]] * entry["floor_count"])

    rng.shuffle(allocated)

    return allocated

In [ ]:
def build_gender_aware_feature_schedule(
    common_variation_data,
    gender_specific_variation_data,
    total_identities_male,
    total_identities_female,
    num_images_per_id,
    seed=42
):
    if "gender" not in common_variation_data:
        raise ValueError("gender.txt is required in common_variation_data")

    n_male = total_identities_male * num_images_per_id
    n_female = total_identities_female * num_images_per_id
    n = n_male + n_female

    common_schedules = {}

    for i, (feature_name, items) in enumerate(common_variation_data.items()):
        if feature_name == "gender":
            continue

        common_schedules[feature_name] = allocate_by_probability(
            items=items,
            n=n,
            seed=seed + i
        )

    rows = []

    gender_blocks = [
        ("male", n_male),
        ("female", n_female),
    ]

    row_idx = 0

    for gender, gender_count in gender_blocks:
        specific_data = gender_specific_variation_data.get(gender, {})

        gender_specific_schedules = {}

        for feature_offset, (feature_name, items) in enumerate(specific_data.items()):
            gender_specific_schedules[feature_name] = allocate_by_probability(
                items=items,
                n=gender_count,
                seed=seed + 1000 + feature_offset
            )

        for local_idx in range(gender_count):
            row = {}

            row["gender"] = gender
            row["gender_text"] = gender
            row["gender_normalized"] = gender

            for feature_name, scheduled_items in common_schedules.items():
                item = scheduled_items[row_idx]
                row[feature_name] = item["label"]
                row[f"{feature_name}_text"] = item["text"]

            for feature_name, scheduled_items in gender_specific_schedules.items():
                item = scheduled_items[local_idx]
                row[feature_name] = item["label"]
                row[f"{feature_name}_text"] = item["text"]

            rows.append(row)
            row_idx += 1

    return rows

In [ ]:
sampled_rows = build_gender_aware_feature_schedule(
    common_variation_data=common_variation_data,
    gender_specific_variation_data=gender_specific_variation_data,
    total_identities_male=identity_counts["male"],
    total_identities_female=identity_counts["female"],
    num_images_per_id=NUM_IMAGES_PER_ID,
    seed=SEED,
)

In [ ]:
def weighted_choice(items, avoid_labels=None):
    avoid_labels = avoid_labels or set()
    candidates = [x for x in items if x["label"] not in avoid_labels]

    if not candidates:
        return None

    labels = candidates
    weights = [x["probability"] for x in candidates]

    return random.choices(labels, weights=weights, k=1)[0]


def add_multi_features(row, variation_data, multi_rules):
    """
    Adds extra weighted items for categories like jewelry or makeup.
    """
    row = dict(row)

    for feature_name, rule in multi_rules.items():
        if not rule.get("enabled", False):
            continue

        if feature_name not in variation_data:
            continue

        current_label = row.get(feature_name)
        current_text = row.get(f"{feature_name}_text")

        selected_labels = []
        selected_texts = []

        if current_label:
            selected_labels.append(current_label)

        if current_text:
            selected_texts.append(current_text)

        max_items = rule.get("max_items", 1)
        chance_extra = rule.get("chance_extra_item", 0.0)
        avoid_labels = set(rule.get("avoid_labels", set()))

        while len(selected_labels) < max_items:
            if random.random() > chance_extra:
                break

            item = weighted_choice(
                variation_data[feature_name],
                avoid_labels=avoid_labels.union(set(selected_labels))
            )

            if item is None:
                break

            selected_labels.append(item["label"])
            selected_texts.append(item["text"])

        row[feature_name] = " + ".join(selected_labels)
        row[f"{feature_name}_text"] = ", ".join(selected_texts)

    return row



sampled_rows = [
    add_multi_features(
        row,
        {**common_variation_data,
         **gender_specific_variation_data.get(row["gender_normalized"], {})},
        MULTI_FEATURE_RULES,
    )
    for row in sampled_rows
]

In [ ]:
def row_to_feature_description(row):
    lines = []

    preferred_order = [
        "gender",
        "pose",
        "lighting",
        "background",
        "clothing",
        "hairstyle",
        "hair_color",
        "headwear",
        "facial_hair",
        "makeup",
        "jewelry",
    ]

    for feature_name in preferred_order:
        text_key = f"{feature_name}_text"

        if text_key in row and pd.notna(row[text_key]):
            lines.append(f"{feature_name}: {row[text_key]}")

    # Include any extra _text fields not listed above
    for key, value in row.items():
        if not key.endswith("_text"):
            continue

        feature_name = key.replace("_text", "")

        if feature_name in preferred_order:
            continue

        if pd.notna(value):
            lines.append(f"{feature_name}: {value}")

    return "\n".join(lines)

In [ ]:
df_samples = pd.DataFrame(sampled_rows)
df_samples["feature_description"] = df_samples.apply(
    lambda row: row_to_feature_description(row.to_dict()), axis=1
)

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("A CUDA-enabled GPU is required for 4-bit Qwen3-14B inference.")

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="cuda",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)

model.eval()

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

pad_token_id = tokenizer.pad_token_id

eos_token_id = tokenizer.eos_token_id
if isinstance(eos_token_id, list):
    eos_token_id = eos_token_id[0]

model.generation_config.pad_token_id = pad_token_id
model.generation_config.eos_token_id = eos_token_id

print("Loaded:", MODEL_NAME)

In [ ]:
SYSTEM_MESSAGE = """
You are an expert prompt writer for realistic image-to-image face portrait editing.
Task: Create ONE concise image-editing prompt using only the provided variation features. 
Requirements:
 - Do not add any extra instructions, constraints, or details beyond the provided features.
 - Do not use the word clear, clearly or similar words which can smoothen the face.
 - Start the prompt directly with the editing instructions based on the features, without any introductory phrases or saying edit this photo or something similar.
"""


USER_TEMPLATE = """
Generate one prompt using the given features.
{features}
Write one clean natural-language prompt.
"""


def clean_generated_prompt(text: str) -> str:
    text = text.strip()

    text = re.sub(
        r"^(assistant|final prompt|prompt)\s*:\s*",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = text.replace("```", "")

    text = re.sub(r"\s+", " ", text)

    return text.strip()


def qwen_generate_prompt(
    feature_description,
    temperature=TEMPERATURE,
    max_tokens=MAX_NEW_TOKENS
):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_MESSAGE.strip()
        },
        {
            "role": "user",
            "content": USER_TEMPLATE.format(
                features=feature_description
            ).strip()
        }
    ]

    try:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False
        )
    except TypeError:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=TOP_P,
            top_k=TOP_K,
            repetition_penalty=REPETITION_PENALTY,
            pad_token_id=pad_token_id,
            eos_token_id=eos_token_id,
            use_cache=True
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]

    text = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    )

    return clean_generated_prompt(text)

In [ ]:
generated_prompts = []
for row in tqdm(df_samples.itertuples(index=False), total=len(df_samples)):
    generated_prompts.append(qwen_generate_prompt(row.feature_description))
df_samples["generated_prompt"] = generated_prompts

In [ ]:
NEGATIVE_PROMPT = """
different person, changed identity, altered face shape, different facial structure,
unrecognizable face, distorted face, deformed eyes, asymmetrical face, harsh shadows,
extreme lighting, extreme pose, face covered, eyes covered, blurry face, low quality,
cartoon, anime, unrealistic skin, heavy makeup, excessive jewelry, large hat shadow,
occluded face, text, logo, watermark
""".replace("\n", " ").strip()

df_samples["negative_prompt"] = NEGATIVE_PROMPT

In [ ]:
for gender, gender_df in df_samples.groupby("gender_normalized"):
    output_file = OUTPUT_DIR / f"face_variation_{gender}.csv"
    gender_df.reset_index(drop=True).to_csv(output_file, index=False, encoding="utf-8")
    print(f"Saved {len(gender_df)} prompts to {output_file}")